# 02 — RAG Pipeline
**DS593 Final Project**

Q&A over Terms of Service and Privacy Policy documents.
Given a plain English question about any app, the system retrieves the most
relevant clause and generates a grounded answer with a citation.


## Setup: Install Dependencies

In [ ]:
!pip install -q langchain-community langchain-openai langchain langchain-text-splitters python-dotenv sentence-transformers chromadb
print("✓ Dependencies installed")


## Step 0: Configure API Key

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY").startswith("sk-"):
    print("✓ OpenAI API key configured")
else:
    raise ValueError("Missing OpenAI API key")


## Step 1: Imports

In [ ]:
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
import json
import os

print("✓ All imports successful")


## Step 2: Load Documents

Load scraped ToS/Privacy documents from the corpus directory.


In [ ]:
CORPUS_DIR = "tos_corpus"

with open(f"{CORPUS_DIR}/manifest.json") as f:
    manifest = json.load(f)

documents = []
for doc in manifest:
    if not os.path.exists(doc["filepath"]):
        continue
    with open(doc["filepath"], encoding="utf-8") as f:
        text = f.read()
    documents.append({
        "text":     text,
        "company":  doc["company"],
        "doc_type": doc["doc_type"],
        "source":   doc["filepath"],
    })

print(f"✓ Loaded {len(documents)} documents")
for d in documents:
    print(f"  {d['company']:15s} | {d['doc_type']:8s} | {len(d['text']):,} chars")


## Step 3: Chunk Documents

Split documents into manageable chunks with overlap to preserve context.
We also store metadata (company, doc_type) alongside each chunk for filtering.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2048,
    chunk_overlap=256,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []
metadatas = []

for doc in documents:
    doc_chunks = text_splitter.split_text(doc["text"])
    for chunk in doc_chunks:
        chunks.append(chunk)
        metadatas.append({
            "company":  doc["company"],
            "doc_type": doc["doc_type"],
            "source":   doc["source"],
        })

print(f"✓ Created {len(chunks)} chunks")
print(f"  Sample: {chunks[0][:100]}...")


## Step 4: Embedding Model

Convert text to vectors so we can compute semantic similarity.
Using `all-MiniLM-L6-v2` — same model as class discussion notebook.


In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

test_embedding = embedding_model.embed_query("What is a Terms of Service?")
print(f"✓ Embedding model loaded")
print(f"  Vector dimension: {len(test_embedding)}")


## Step 5: Vector Database

Index all chunks for fast similarity search.


In [ ]:
vector_db = Chroma.from_texts(
    texts=chunks,
    embedding=embedding_model,
    metadatas=metadatas,
    persist_directory="./chroma_db_rag"
)

print(f"✓ Vector database created")
print(f"  Indexed: {len(chunks)} chunks")


## Step 6: Initialize LLM

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    max_tokens=500
)

print("✓ LLM initialized")


## Step 7: Retrieval Function

Retrieve the top-k most relevant chunks for a query.
Filters by company so we only retrieve from the right documents.


In [ ]:
def retrieve_chunks(query: str, company: str, k: int = 3) -> List[str]:
    """Retrieve top-k most relevant chunks for a query, filtered by company."""
    results = vector_db.similarity_search(
        query,
        k=k,
        filter={"company": company}
    )
    return [doc.page_content for doc in results]


# Test it
test_chunks = retrieve_chunks("Can Spotify sell my data?", company="spotify", k=2)
print(f"Retrieved {len(test_chunks)} chunks")
print(f"  Sample: {test_chunks[0][:150]}...")


## Step 8: RAG Query Function

Complete RAG pipeline: retrieve relevant chunks → build prompt → generate answer.


In [ ]:
def rag_query(question: str, company: str, k: int = 3) -> dict:
    """Complete RAG pipeline — retrieve then generate."""
    try:
        retrieved_chunks = retrieve_chunks(question, company=company, k=k)
        context = "\n\n".join(retrieved_chunks)

        prompt = f"""You are a legal analyst helping users understand Terms of Service and Privacy Policy documents.
Answer the question based ONLY on the provided document excerpts.
Be direct and specific. If the answer is not in the excerpts, say so.
Cite the exact clause that supports your answer.

Document excerpts:
{context}

Question: {question}

Answer:"""

        answer = llm.invoke(prompt).content

        return {
            "question": question,
            "company":  company,
            "chunks":   retrieved_chunks,
            "answer":   answer,
        }

    except Exception as e:
        print(f"Error: {e}")
        return {"error": str(e)}


# Test it
result = rag_query("Can Spotify sell my personal data to third parties?", company="spotify")
print(f"Question: {result['question']}")
print(f"\nAnswer: {result['answer']}")


## Step 9: Baseline Function

Vanilla LLM with no retrieval — for comparison in evaluation.


In [ ]:
def baseline_query(question: str, company: str) -> str:
    """Vanilla LLM with no retrieval."""
    prompt = f"""You are a legal analyst. Answer this question about {company}'s Terms of Service or Privacy Policy.
Be direct and specific. If you are not sure, say so.

Question: {question}

Answer:"""
    return llm.invoke(prompt).content


# Test side by side
question = "How many days does Discord give you to opt out of arbitration?"
company  = "discord"

print("=" * 70)
print("RAG ANSWER (grounded in actual document)")
print("=" * 70)
result = rag_query(question, company=company)
print(result["answer"])

print("\n" + "=" * 70)
print("BASELINE ANSWER (no retrieval)")
print("=" * 70)
print(baseline_query(question, company=company))


## Step 10: More Test Queries

In [ ]:
tests = [
    ("What is the minimum age to use Tinder?", "tinder"),
    ("What is the maximum amount X will pay in a lawsuit?", "twitter_x"),
    ("How many days does PayPal give personal account holders before changes that reduce rights?", "paypal"),
    ("What is the minimum age to use LinkedIn?", "linkedin"),
]

for question, company in tests:
    print("=" * 70)
    print(f"COMPANY:  {company.upper()}")
    print(f"QUESTION: {question}")
    print("=" * 70)
    result = rag_query(question, company=company)
    print(f"RAG: {result['answer'][:300]}")
    print(f"\nBASELINE: {baseline_query(question, company)[:300]}")
    print()
